# text2sql-forge: fine-tune Qwen2.5-Coder-1.5B for text-to-SQL

Runs the whole experiment on a **free 16 GB T4** (Kaggle: *Settings → Accelerator → GPU T4*; Colab: *Runtime → Change runtime type → T4 GPU*):

1. Build training data (Spider train + synthetic augmentation) with database-disjoint splits
2. Evaluate the **base** model (execution accuracy on Spider dev)
3. QLoRA fine-tune (~1–2 h on a T4 for 2 epochs)
4. Evaluate the **fine-tuned** model
5. Paired statistical comparison with `llm-verdict`

Everything below writes to `/kaggle/working` (or `/content` on Colab) so results survive the session.

## 0. Setup
The repos are public, so no token is needed. (If you ever make them private, add a GitHub token as a Kaggle secret named `GH_TOKEN`.)


In [ ]:
import os, subprocess, pathlib
# Kaggle's "GPU T4 x2" exposes two GPUs; 4-bit QLoRA training is simplest and most reliable on one.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
WORK = pathlib.Path("/kaggle/working" if os.path.exists("/kaggle") else "/content")
os.chdir(WORK)

token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GH_TOKEN")
except Exception:
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
    except Exception:
        pass
auth = f"{token}@" if token else ""
for repo in ["text2sql-forge", "llm-verdict"]:
    if not (WORK / repo).exists():
        subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/bharat-sing-h/{repo}"], check=True)

In [ ]:
os.chdir(WORK / "text2sql-forge")
!pip install -q -e ".[train]" -e ../llm-verdict
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Data
**Spider** (Yu et al., 2018; CC BY-SA 4.0): download `spider_data.zip` from the official page (https://yale-lily.github.io/spider), then either attach it as a Kaggle dataset (*Add Input*). The notebook finds the folder containing `train_spider.json` automatically, or unzip it into `data/spider`.

If Spider is not found, the notebook falls back to the synthetic benchmark so every cell still runs.

In [ ]:
import pathlib
# Look for Spider in attached Kaggle datasets (any folder containing train_spider.json)
found = sorted(pathlib.Path("/kaggle/input").rglob("train_spider.json")) if pathlib.Path("/kaggle/input").exists() else []
SPIDER_ROOT = found[0].parent if found else pathlib.Path(os.environ.get("SPIDER_ROOT", "data/spider"))
print("SPIDER_ROOT =", SPIDER_ROOT)
HAVE_SPIDER = (SPIDER_ROOT / "train_spider.json").exists()
print("Spider found:", HAVE_SPIDER)

!t2s data synth --out data/synthetic --instances 6 --per-db 60
if HAVE_SPIDER:
    !t2s data build --source spider --root {SPIDER_ROOT} --add-synthetic data/synthetic --out data
    EVAL_SET, DB_ROOT = "spider", str(SPIDER_ROOT / "database")
else:
    !t2s data build --source synthetic --root data/synthetic --out data
    EVAL_SET, DB_ROOT = "synthetic", "data/synthetic/databases"
!cat data/DATA_CARD.json

## 2. Baseline: the base model, no fine-tuning
`LIMIT` keeps this quick; use the full dev set (1,034 questions) for your final numbers.

In [ ]:
BASE = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
LIMIT = 400
%env T2S_MODEL={BASE}
%env EVAL_SET={EVAL_SET}
%env DB_ROOT={DB_ROOT}
%env LIMIT={LIMIT}
%env RUN_NAME=base
!llmverdict run configs/eval_local.yaml --label base

## 3. QLoRA fine-tuning

In [ ]:
!t2s train configs/qlora_qwen2.5-coder-1.5b.yaml
!python -c "import json; i=json.load(open('outputs/qwen2.5-coder-1.5b-sql/run_info.json')); print({k: i[k] for k in ['train_examples','val_examples','train_loss','eval_loss','trainable_params','train_runtime_s','device']})"

## 4. Evaluate the fine-tuned model (same questions, same settings)

In [ ]:
# each `!llmverdict` call is a fresh process, so the new env vars select the fine-tuned model
%env T2S_ADAPTER=outputs/qwen2.5-coder-1.5b-sql/adapter
%env RUN_NAME=finetuned
!llmverdict run configs/eval_local.yaml --label finetuned

## 5. Is the improvement real?
Paired comparison on identical questions: exact McNemar test + bootstrap CI on the difference, plus per-difficulty breakdown.

In [ ]:
!llmverdict compare base__{EVAL_SET} finetuned__{EVAL_SET} -o reports/comparison.md
!llmverdict report finetuned__{EVAL_SET} -o reports/finetuned.md
from IPython.display import Markdown, display
display(Markdown(open("reports/comparison.md").read()))

## 6. (Optional) Merge and publish for serving
Merged weights can be served by vLLM (`deploy/docker-compose.yml`) or converted to GGUF for Ollama.

In [ ]:
!t2s merge outputs/qwen2.5-coder-1.5b-sql/adapter outputs/merged
# from huggingface_hub import HfApi; HfApi().upload_folder(folder_path="outputs/merged", repo_id="<you>/qwen2.5-coder-1.5b-sql")

In [ ]:
# keep the evidence: results DB, reports, training metadata
!mkdir -p {WORK}/results && cp -r .llmverdict reports outputs/qwen2.5-coder-1.5b-sql/run_info.json data/DATA_CARD.json {WORK}/results/